In [1]:
import os
import io
from pydantic import BaseModel, ValidationError
from typing import List, Literal
from openai import OpenAI
from promptic import llm
from tenacity import retry, retry_if_exception_type
from dotenv import load_dotenv
import concurrent.futures as cf
from tempfile import NamedTemporaryFile

In [9]:
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

In [10]:
text = """
I opened my eyes upon a strange and weird landscape. I knew that I was                                                                                                                                     
on Mars; not once did I question either my sanity or my wakefulness. I                                                                                                                                     
was not asleep, no need for pinching here; my inner consciousness told                                                                                                                                     
me as plainly that I was upon Mars as your conscious mind tells you                                                                                                                                        
that you are upon Earth. You do not question the fact; neither did I.                                                                                                                                      
                                                                                                                                                                                                           
I found myself lying prone upon a bed of yellowish, mosslike vegetation                                                                                                                                    
which stretched around me in all directions for interminable miles. I                                                                                                                                      
seemed to be lying in a deep, circular basin, along the outer verge of                                                                                                                                     
which I could distinguish the irregularities of low hills.                                                                                                                                                 
                                                                                                                                                                                                           
It was midday, the sun was shining full upon me and the heat of it was                                                                                                                                     
rather intense upon my naked body, yet no greater than would have been                                                                                                                                     
true under similar conditions on an Arizona desert. Here and there were                                                                                                                                    
slight outcroppings of quartz-bearing rock which glistened in the                                                                                                                                          
sunlight; and a little to my left, perhaps a hundred yards, appeared a                                                                                                                                     
low, walled enclosure about four feet in height. No water, and no other                                                                                                                                    
vegetation than the moss was in evidence, and as I was somewhat thirsty                                                                                                                                    
I determined to do a little exploring.                                                                                                                                                                     
                                                                                                                                                                                                           
Springing to my feet I received my first Martian surprise, for the                                                                                                                                         
effort, which on Earth would have brought me standing upright, carried                                                                                                                                     
me into the Martian air to the height of about three yards. I alighted                                                                                                                                     
softly upon the ground, however, without appreciable shock or jar. Now                                                                                                                                     
commenced a series of evolutions which even then seemed ludicrous in                                                                                                                                       
the extreme. I found that I must learn to walk all over again, as the                                                                                                                                      
muscular exertion which carried me easily and safely upon Earth played                                                                                                                                     
strange antics with me upon Mars. 
"""

In [11]:
class DialogueItem(BaseModel):
    text: str
    speaker: Literal["female-1", "male-1", "female-2"]

    @property
    def voice(self):
        return {
            "female-1": "alloy",
            "male-1": "onyx",
            "female-2": "shimmer",
        }[self.speaker]


class Dialogue(BaseModel):
    scratchpad: str
    dialogue: List[DialogueItem]

In [18]:
@retry(retry=retry_if_exception_type(ValidationError))
@llm(model="gpt-4o-mini",)
def generate_dialogue(text: str) -> Dialogue:
    """
    Your task is to take the input text provided and turn it into an engaging, informative podcast dialogue. The input text may be messy or unstructured, as it could come from a variety of sources like PDFs or web pages. Don't worry about the formatting issues or any irrelevant information; your goal is to extract the key points and interesting facts that could be discussed in a podcast.

    Here is the input text you will be working with:

    <input_text>
    {text}
    </input_text>

    First, carefully read through the input text and identify the main topics, key points, and any interesting facts or anecdotes. Think about how you could present this information in a fun, engaging way that would be suitable for an audio podcast.

    <scratchpad>
    Brainstorm creative ways to discuss the main topics and key points you identified in the input text. Consider using analogies, storytelling techniques, or hypothetical scenarios to make the content more relatable and engaging for listeners.

    Keep in mind that your podcast should be accessible to a general audience, so avoid using too much jargon or assuming prior knowledge of the topic. If necessary, think of ways to briefly explain any complex concepts in simple terms.

    Use your imagination to fill in any gaps in the input text or to come up with thought-provoking questions that could be explored in the podcast. The goal is to create an informative and entertaining dialogue, so feel free to be creative in your approach.

    Write your brainstorming ideas and a rough outline for the podcast dialogue here. Be sure to note the key insights and takeaways you want to reiterate at the end.
    </scratchpad>

    Now that you have brainstormed ideas and created a rough outline, it's time to write the actual podcast dialogue. Aim for a natural, conversational flow between the host and any guest speakers. Incorporate the best ideas from your brainstorming session and make sure to explain any complex topics in an easy-to-understand way.

    <podcast_dialogue>
    Write your engaging, informative podcast dialogue here, based on the key points and creative ideas you came up with during the brainstorming session. Use a conversational tone and include any necessary context or explanations to make the content accessible to a general audience. Use made-up names for the hosts and guests to create a more engaging and immersive experience for listeners. Do not include any bracketed placeholders like [Host] or [Guest]. Design your output to be read aloud -- it will be directly converted into audio.

    Make the dialogue as long and detailed as possible, while still staying on topic and maintaining an engaging flow. Aim to use your full output capacity to create the longest podcast episode you can, while still communicating the key information from the input text in an entertaining way.

    At the end of the dialogue, have the host and guest speakers naturally summarize the main insights and takeaways from their discussion. This should flow organically from the conversation, reiterating the key points in a casual, conversational manner. Avoid making it sound like an obvious recap - the goal is to reinforce the central ideas one last time before signing off.
    </podcast_dialogue>
    """
   
llm_output = generate_dialogue(text)


In [19]:
llm_output

Dialogue(scratchpad="In this podcast episode, we will explore the intriguing idea of experiencing life on Mars. The narrative involves the protagonist waking up on this alien planet, depicting the landscape and sensations vividly. Major points include the new physical challenges posed by Mars's lower gravity, the unusual environment of yellowish vegetation, and the initial surprises that come with adapting to this new world. We’ll discuss how the character learns to walk again, highlighting the differences between life on Earth and Mars. We could introduce fun hypothetical scenarios such as: If you could travel to Mars, what would you want to experience first? How would you adapt your everyday activities in such a unique environment?\n\nThroughout the episode, we will engage in storytelling techniques and use humor to describe potential first experiences on Mars, eliciting the audience's imagination. We will also reflect on what this narrative reveals about our innate curiosity and the

In [23]:
def get_mp3(text: str, voice: str, api_key: str = None) -> bytes:
    client = OpenAI(
        api_key=api_key or os.getenv("OPENAI_API_KEY"),
    )

    with client.audio.speech.with_streaming_response.create(
        model="tts-1",
        voice=voice,
        input=text,
    ) as response:
        with io.BytesIO() as file:
            for chunk in response.iter_bytes():
                file.write(chunk)
            return file.getvalue()

In [24]:
audio = b""
transcript = ""

characters = 0

with cf.ThreadPoolExecutor() as executor:
    futures = []
    for line in llm_output.dialogue:
        transcript_line = f"{line.speaker}: {line.text}"
        future = executor.submit(get_mp3, line.text, line.voice, api_key)
        futures.append((future, transcript_line))
        characters += len(line.text)

    for future, transcript_line in futures:
        audio_chunk = future.result()
        audio += audio_chunk
        transcript += transcript_line + "\n\n"

In [26]:
temporary_directory = "./output/"
os.makedirs(temporary_directory, exist_ok=True)

temporary_file = NamedTemporaryFile(
    dir=temporary_directory,
    delete=False,
    suffix=".mp3",
)
temporary_file.write(audio)
temporary_file.close()